# Task 3: Sequence Classification

## Long Short-Term Memory (LSTM)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)


### Load data

In [3]:
X_train_w = np.load('../data/3_processed/X_train_w.npy')
y_train_w = np.load('../data/3_processed/y_train_w.npy')
X_test_w = np.load('../data/3_processed/X_test_w.npy')
y_test_w = np.load('../data/3_processed/y_test_w.npy')

### Encode Labels

In PyTorch, the `CrossEntropyLoss` function for training the model does not accept text ("N", "S", "V"). It requires that the target labels be integer indices, starting from 0.

In [4]:
to_encode = {"N": 0, "S": 1, "V": 2}
if y_train_w.dtype.kind in {'U', 'S', 'O'}:
    y_train_w = np.vectorize(to_encode.get)(y_train_w)
if y_test_w.dtype.kind in {'U', 'S', 'O'}:
    y_test_w = np.vectorize(to_encode.get)(y_test_w)
    
print(f"Unique train labels: {np.unique(y_train_w)}")
print(f"Unique test labels: {np.unique(y_test_w)}")

Unique train labels: [0 1 2]
Unique test labels: [0 1 2]


### PyTorch Tensors

`y` contains the classification labels. After mapping, these are integers with no decimals and represent discrete categories. Thes numbers are not mathematical values; they are indices (`torch.long`).

In [5]:
X_train_tensor = torch.tensor(X_train_w, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_w, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_w, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_w, dtype=torch.long)

print(f"X_train_tensor shape: {X_train_tensor.shape}")
print(f"y_train_tensor shape: {y_train_tensor.shape}")


X_train_tensor shape: torch.Size([24027, 10, 8])
y_train_tensor shape: torch.Size([24027])


### PyTorch Dataset and DataLoader

In [6]:
class ECGSequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
train_dataset = ECGSequenceDataset(X_train_tensor, y_train_tensor)
test_dataset = ECGSequenceDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

A `batch_size` of 64 provides an average that is good enough to point the model in the right direction, but it retains a small amount of noise from one batch to the next. This noise is beneficial because it acts as a natural regularizer and helps the network generalize better to the test data.

The training tensor has  24,027 training examples:
$$
\frac{24,027}{64} \approx 375
$$
This means that in each training epoch, the model will adjust its weights 375 times.

### LSTM Architecture Definition

In [7]:
class LSTMClassifier(nn.Module):
    def __init__(
            self,
            input_size=8,
            hidden_size=64,
            num_layers=1,
            num_classes=3
    ):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        lstm_output, (hidden, cell) = self.lstm(x)
        last_output = lstm_output[:, -1, :]  # Take the last time step's output
        output = self.fc(last_output)
        return output